In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

from quantify_core.data.handling import set_datadir, load_dataset

import os
DATA_DIR = os.environ.get("QBLOX_DATADIR", "CD1")  # quantify data dir; override with QBLOX_DATADIR
set_datadir(DATA_DIR)  # change me!

FIGDIR = 'paper_plots'
os.makedirs(FIGDIR, exist_ok=True)

import sys
from tqdm.auto import tqdm
from snl_qblox.helpers import *
from snl_qblox.tomography_tools import *

import lmfit
from quantify_core.analysis.fitting_models import ResonatorModel, LorentzianModel, ExpDecayModel, DecayOscillationModel
resonator_model = ResonatorModel()
lorentzian_model = LorentzianModel()
exp_decay_model = ExpDecayModel()
decay_osci_model = DecayOscillationModel()

from snl_qblox.analyzer import BeatingDecayOscillationModel, DoubleLorentzianModel, TripleLorentzianModel, SwapDecayModel
beating_model = BeatingDecayOscillationModel()
double_lorentzian_model = DoubleLorentzianModel()
triple_lorentzian_model = TripleLorentzianModel()
swap_decay_model = SwapDecayModel()

# PRA single-column figure style: 3.375 in wide, 8/7 pt type, vector output.
# Defined once here and reused (via plt.rc_context(PRA)) by every PRA-format figure below.
PRA = {
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.linewidth': 0.7,
    'axes.edgecolor': 'black',
    'axes.spines.top': True,       # papers keep the full frame
    'axes.spines.right': True,
    'axes.grid': False,
    'xtick.color': 'black',
    'ytick.color': 'black',
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'xtick.minor.size': 1.8,
    'ytick.minor.size': 1.8,
    'lines.linewidth': 0.8,
    'savefig.dpi': 600,
    'pdf.fonttype': 42,            # embed as TrueType, not Type 3
    'ps.fonttype': 42,
}

def curr2flux(I, half_period = 0.8711, offset = 0.0055):
    I = np.array(I)
    return (I - offset) / (2 * half_period)


def flux2curr(flux, half_period = 0.8711, offset = 0.0055):
    flux = np.array(flux)
    return flux * (2 * half_period) + offset

# Path to the sim/theory notebook on the other machine (synced via Drive) --
# used to pull in its finished figures (see the "Theory figures" section) and,
# below, to reuse its plotting code with data read straight from its results/ folder.
REMOTE_SIM_DIR = os.environ.get("SNL315_SIM_DIR") or (r"C:\Users\1238Tech\Google Drive Streaming\Other computers"
                  r"\My MacBook Pro\Research\Palmer Lab\Python Codes"
                  r"\SNAIL Two Qubit Gate\Zhuo's sim\SNL315")

# Colour palette from SNAIL_315_plots.ipynb, copied verbatim so figures that combine
# data from both notebooks use identical colours for the same series.
C = {'Q1': '#0072B2',      # blue
     'Q2': '#D55E00',      # vermillion
     'SNAIL': '#009E73',   # green
     'ZZ': '#8B5CF6'}      # violet
GRID = '#d4d4d8'

## 1. iSWAP Chevron

In [ ]:
dataset = load_dataset('20260910-233949')
x0_data = np.unique(dataset.x0)
x1_data = np.unique(dataset.x1)
y0_data = dataset.y0.values.reshape(x1_data.size, x0_data.size)
y1_data = dataset.y1.values.reshape(x1_data.size, x0_data.size)


def draw_iswap_chevron(ax_q1, ax_q2, x0_data, x1_data, y0_data, y1_data):
    """Draw the Q1/Q2 iSWAP population chevrons onto the given axes and return the
    pcolormesh (the caller attaches its own colorbar, since that placement is figure-
    specific). Shared by this section's standalone figure and the combined figure in
    section 5, so the two never drift apart in style."""
    for ax, y_data, lab in [(ax_q1, y0_data, 'Q1'), (ax_q2, y1_data, 'Q2')]:
        pcm = ax.pcolormesh(x0_data * 1e6, x1_data / 1e6, y_data, shading='auto',
                            vmin=0, vmax=1, rasterized=True)
        # No titles: each panel is named by a legend instead. A pcolormesh has no series to
        # key, so the handle is a zero-length invisible line and handlelength=0 collapses it
        # -- the entry is the label alone inside a legend frame. The opaque white ground keeps
        # it readable where it sits over the dark end of the colormap.
        leg = ax.legend([plt.Line2D([], [], lw=0)], [lab], loc='upper left',
                        handlelength=0, handletextpad=0, borderpad=0.35, frameon=True,
                        facecolor='white', edgecolor='gray', framealpha=0.9)
        leg.get_frame().set_linewidth(0.5)
        # Only the labelled major ticks are kept (no unlabelled minor ticks).
        ax.set_xticks([0, 1, 2, 3])
        ax.tick_params(which='major', direction='in', color='gray')
    ax_q1.set_ylabel(r'$\omega_p/2\pi$ (MHz)', labelpad=1)
    ax_q1.set_yticks([256, 258, 260])
    return pcm


# The panel box is placed by hand instead of by tight_layout, and the colorbar gets its own
# axes, so the two panels keep a fixed width no matter what the colorbar labels do. That is
# what lets the single x label be centred on the panel PAIR -- x = (LEFT + RIGHT) / 2 -- rather
# than on the figure, whose centre the colorbar would pull to the right.
# For a two-column figure, 3.375 -> 6.75 below is the only number that has to change.
LEFT, RIGHT, BOTTOM, TOP = 0.135, 0.80, 0.24, 0.96

with plt.rc_context(PRA):
    # sharey: one frequency axis for both panels. The right panel's tick labels are dropped
    # automatically, so a single y label on the left panel serves the pair -- and unlike
    # set_yticks([]) it keeps the tick marks themselves, which the frame needs.
    fig, axes = plt.subplots(1, 2, figsize=(3.375, 1.7), sharey=True,
                             gridspec_kw={'wspace': 0.07})
    fig.subplots_adjust(left=LEFT, right=RIGHT, bottom=BOTTOM, top=TOP)

    pcm = draw_iswap_chevron(axes[0], axes[1], x0_data, x1_data, y0_data, y1_data)
    fig.supxlabel(r'Pump duration (μs)', x=0.5 * (LEFT + RIGHT), y=0.06)

    # Both panels share vmin/vmax, so either mappable describes the colorbar; pcm is the
    # last one drawn. direction='out' because 'in' would put the ticks on the bar itself.
    cax = fig.add_axes([RIGHT + 0.025, BOTTOM, 0.025, TOP - BOTTOM])
    cb = fig.colorbar(pcm, cax=cax, ticks=[0, 0.5, 1])
    cb.ax.set_yticklabels(['0', '0.5', '1'])
    cb.set_label('Population')
    cb.outline.set_linewidth(0.7)
    cb.ax.tick_params(which='both', direction='out', color='black', left=False, right=True)

    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 'iswap_population.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

In [ ]:
plt.plot(dBm2W(x2_data), iswap_freq_list)

In [ ]:
iswap_time_list = np.array([4445.43344161, 2487.46679019, 1769.69022107, 1355.17052456,
       1111.8289699 ,  910.56320898,  779.8343058 ,  692.63575305,
        612.22937625,  558.98654983,  510.7008313 ,  465.95058917,
        420.13255892,  403.21160312,  368.88586215,  343.52911985,
        325.67063241,  303.12693932,  283.74772871,  267.21571096,
        251.45581596,  236.33778588,  225.1551897 ,  218.64193025,
        211.6888062 ,  196.81780481,  183.81658517,  170.78610428,
        166.1010032 ,  154.19578608,  147.8305775 ,  139.7103845 ,
        131.56673958,  121.08294428,  117.30251157, 113.52207887,
        104.34323636,  100.67192871,  92.65583148,  90.9009213 ,
        87.83316129])

iswap_freq_list = np.array([2.57841320e+08, 2.57858086e+08, 2.57903171e+08, 2.57943725e+08,
       2.57993973e+08, 2.58042972e+08, 2.58129867e+08, 2.58199218e+08,
       2.58305000e+08, 2.58403403e+08, 2.58523111e+08, 2.58667170e+08,
       2.58791172e+08, 2.58952353e+08, 2.59153246e+08, 2.59278024e+08,
       2.59447054e+08, 2.59627538e+08, 2.59785044e+08, 2.59998020e+08,
       2.60162258e+08, 2.60306093e+08, 2.60458992e+08, 2.60811467e+08,
       2.61145414e+08, 2.61376761e+08, 2.61585393e+08, 2.61704010e+08,
       2.61743774e+08, 2.62079959e+08, 2.62675422e+08, 2.62750771e+08,
       2.62890369e+08, 2.62997539e+08, 2.63321326e+08, 2.63645113e+08,
       2.63819999e+08, 2.64246641e+08, 2.64577812e+08, 2.64880802e+08,
       2.65301651e+08])

In [ ]:
pump_power_list = W2dBm(np.linspace(0.001, 0.03, 21))
pump_freq_list = np.array([2.57974842e+08, 2.58084210e+08, 2.58246706e+08, 2.58390314e+08,
       2.58486658e+08, 2.58622774e+08, 2.58777517e+08, 2.58939475e+08,
       2.59051136e+08, 2.59155342e+08, 2.59272327e+08, 2.59392023e+08,
       2.59524625e+08, 2.59574192e+08, 2.59752274e+08, 2.59870537e+08,
       2.59959175e+08, 2.60126662e+08, 2.60248960e+08, 2.60344522e+08,
       2.60473845e+08])

t_swap_list = np.array([1430.89534908, 899.04974534, 714.4102194, 604.20519246,
       533.35797234, 481.44829084, 442.88699757, 415.38642205,
       387.11145324, 364.26370528, 344.07038045, 326.32759552,
       313.8624228, 297.52544188, 285.64785854, 275.9350227,
       262.39357591, 255.54257086, 247.16073838, 239.15164503,
       231.68860879])/1e9

In [ ]:
pump_power_list = Vp2dBm(np.linspace(dBm2Vp(-10), dBm2Vp(20), 41))

pump_freq_list = np.array([257.88335217, 257.90480367, 257.94369833, 257.991501  ,
       258.03655533, 258.09602733, 258.1756835 , 258.255225  ,
       258.36727517, 258.47074817, 258.57771217, 258.69827117,
       258.83780217, 258.9888795 , 259.1474075 , 259.30971983,
       259.44610983, 259.64961983, 259.8075365 , 260.012115  ,
       260.18938   , 260.39901583, 260.586329  , 260.85980233,
       261.04176817, 261.30920633, 261.547754  , 261.71493133,
       262.00186167, 262.16416783, 262.41684417, 262.68073917,
       262.92895383, 263.07489133, 263.33942167, 263.5312455 ,
       263.88962067, 264.14802033, 264.40573067, 264.6878145 ,
       264.909075  ]) * 1e6

t_swap_list = np.array([4426.97658279, 2563.58652216, 1708.96371655, 1344.22095463,
       1104.5731608 ,  920.92825288,  797.94072496,  696.70920826,
        620.77711872,  556.913166  ,  505.9761251 ,  465.41804944,
        429.05511363,  399.02049024,  370.45010611,  342.4734359 ,
        322.05460042,  305.38546271,  284.57422493,  267.02051405,
        252.45484819,  237.55218837,  225.0971286 ,  213.52435847,
        200.80296006,  192.06816168,  183.8921971 ,  171.89646769,
        164.31117555,  155.76140157,  146.72249793,  138.68205754,
        130.80579647,  124.3922651 ,  118.630163  ,  111.57194808,
        106.51149304,   99.84176219,   96.1109823 ,   90.1840226 ,
         85.86851419])  / 1e9

In [ ]:
plt.plot(dBm2W(pump_power_list), temp_iswap_freq_list/6)

In [ ]:
# temp_iswap_freq_list = np.copy(iswap_freq_list)
# temp_iswap_time_list = np.copy(iswap_time_list)
temp_iswap_freq_list += iswap_freq_list
temp_iswap_time_list += iswap_time_list

In [ ]:
# '20260928-182140'
# '20260928-205543'
# '20260928-211142'
# '20260928-212714'
# '20260928-214742'
# '20260928-220243'

dataset = load_dataset('20260928-220243')
x2_data = np.unique(dataset.x2)

iswap_time_list = np.zeros(x2_data.size)
iswap_freq_list = np.zeros(x2_data.size)

for i in range(len(x2_data)):
    temp_dataset = dataset.where(dataset.x2==x2_data[i], drop=True)
    iswap_freq = iswap_chevron_fit(temp_dataset)
    iswap_freq_list[i] = iswap_freq

    x0_data = np.unique(temp_dataset.x0)
    x1_data = np.unique(temp_dataset.x1)
    freq_idx = np.argmin(np.abs(x1_data - iswap_freq))
    temp_y0_data = temp_dataset.y0.values.reshape(x1_data.size, x0_data.size)[freq_idx]

    guess = swap_decay_model.guess(temp_y0_data, t=x0_data)
    guess['tau_osc'].value = 20e-6
    fit_result = swap_decay_model.fit(temp_y0_data, params=guess, t=x0_data)
    iswap_time_list[i] = 1/fit_result.params["frequency"].value*1e9/2

In [ ]:
dataset

## 2. Chevron vs Power

In [ ]:
# 20260922-001556

dataset = load_dataset('20260911-020820')
x0_data = np.unique(dataset.x0)
x1_data = np.unique(dataset.x1)
x2_data = np.unique(dataset.x2)

iswap_time_list = np.zeros(x2_data.size)
iswap_freq_list = np.zeros(x2_data.size)

for i in range(len(x2_data))[:18]:
    temp_dataset = dataset.where(dataset.x2==x2_data[i], drop=True)
    iswap_freq = iswap_chevron_fit(temp_dataset)
    iswap_freq_list[i] = iswap_freq

    freq_idx = np.argmin(np.abs(x1_data - iswap_freq))
    temp_y0_data = temp_dataset.y0.values.reshape(x1_data.size, x0_data.size)[freq_idx]

    guess = swap_decay_model.guess(temp_y0_data, t=x0_data)
    guess['tau_osc'].value = 20e-6
    fit_result = swap_decay_model.fit(temp_y0_data, params=guess, t=x0_data)
    iswap_time_list[i] = 1/fit_result.params["frequency"].value*1e9/2

dataset1 = load_dataset('20260911-164905')
x0_data1 = np.unique(dataset1.x0)
x1_data1 = np.unique(dataset1.x1)
x2_data1 = np.unique(dataset1.x2)

for i in range(len(x2_data1)):
    temp_dataset = dataset1.where(dataset1.x2==x2_data1[i], drop=True)
    iswap_freq = iswap_chevron_fit(temp_dataset)
    iswap_freq_list[i+18] = iswap_freq

    freq_idx = np.argmin(np.abs(x1_data1 - iswap_freq))
    temp_y0_data = temp_dataset.y0.values.reshape(x1_data1.size, x0_data.size)[freq_idx]

    guess = swap_decay_model.guess(temp_y0_data, t=x0_data1)
    guess['tau_osc'].value = 20e-6
    fit_result = swap_decay_model.fit(temp_y0_data, params=guess, t=x0_data1)
    iswap_time_list[i+18] = 1/fit_result.params["frequency"].value*1e9/2

In [ ]:
plt.plot(dBm2W(x2_data), iswap_time_list, 'o-')

## 3. T1 and T2

### 3.1 Overtime

In [ ]:
multi_t1_and_t2_tuid_list = ['20260913-142903','20260913-145548','20260913-152233','20260913-154918',
'20260913-161603','20260913-164248','20260913-170934','20260913-173619','20260913-180304','20260913-182949',
'20260913-185634','20260913-192320','20260913-195005','20260913-201651','20260913-204337','20260913-211022',
'20260913-213707','20260913-220352','20260913-223038','20260913-225723','20260913-232409','20260913-235054',
'20260914-001740','20260914-004425','20260914-011109','20260914-013754','20260914-020440','20260914-023126',
'20260914-025811','20260914-032456','20260914-035141','20260914-041827','20260914-044513','20260914-051159',
'20260914-053845','20260914-060530','20260914-063215','20260914-065901','20260914-072547','20260914-075232',
'20260914-081918','20260914-084603','20260914-091249','20260914-093936','20260914-100621','20260914-103306',
'20260914-105951','20260914-112637','20260914-115322','20260914-122008']

In [ ]:
q1_t1_list = []
q1_t2_list = []
q1_t2_ram_list = []
q1_t2_ram_freq_list = []

q2_t1_list = []
q2_t2_list = []
q2_t2_ram_list = []
q2_t2_ram_freq_list = []

for tuid in tqdm(multi_t1_and_t2_tuid_list):
    [q1_t1, q1_t2, q1_t2_ram, q1_t2_ram_freq], [q2_t1, q2_t2, q2_t2_ram, q2_t2_ram_freq] = t1_and_t2_fit(tuid, r_squared=0.6)
    q1_t1_list.append(q1_t1)
    q1_t2_list.append(q1_t2)
    q1_t2_ram_list.append(q1_t2_ram)
    q1_t2_ram_freq_list.append(q1_t2_ram_freq)

    q2_t1_list.append(q2_t1)
    q2_t2_list.append(q2_t2)
    q2_t2_ram_list.append(q2_t2_ram)
    q2_t2_ram_freq_list.append(q2_t2_ram_freq)

q1_t1_list_full = np.array(q1_t1_list).flatten()
q1_t2_list_full = np.array(q1_t2_list).flatten()
q1_t2_ram_list_full = np.array(q1_t2_ram_list).flatten()
q1_t2_ram_freq_list_full = np.array(q1_t2_ram_freq_list).flatten()

q2_t1_list_full = np.array(q2_t1_list).flatten()
q2_t2_list_full = np.array(q2_t2_list).flatten()
q2_t2_ram_list_full = np.array(q2_t2_ram_list).flatten()
q2_t2_ram_freq_list_full = np.array(q2_t2_ram_freq_list).flatten()

In [ ]:
q1_t1_list = q1_t1_list_full[740:]
q1_t2_list = q1_t2_list_full[740:]
q1_t2_ram_list = q1_t2_ram_list_full[740:]

q2_t1_list = q2_t1_list_full[740:]
q2_t2_list = q2_t2_list_full[740:]
q2_t2_ram_list = q2_t2_ram_list_full[740:]

In [ ]:
# test_ds = load_dataset(multi_t1_and_t2_tuid_list[32])
# test_ds = test_ds.where(test_ds.x1 == 2, drop=True)

In [ ]:
BIN = 10
Time = 16.3
n_bins = q1_t1_list.reshape(-1, BIN).shape[0]
x = np.linspace(0, Time, n_bins)

series = [
    (q1_t1_list, '$T_1$', 'C0'),
    (q1_t2_ram_list, '$T_2^*$', 'C1'),
    (q1_t2_list, '$T_2^E$', 'C2'),
]

with plt.rc_context(PRA):
    fig, ax = plt.subplots(figsize=(3.375, 2.1))
    for data, label, color in series:
        binned = data.reshape(-1, BIN)
        mean = np.nanmean(binned, axis=1)
        std = np.nanstd(binned, axis=1)
        ax.fill_between(x, mean - std, mean + std, alpha=0.25, color=color, linewidth=0)
        ax.plot(x, mean, color=color, label=label)

    ax.set_xlim(0, Time)
    ax.set_ylim(0, 100)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Q1 coherence time (μs)')
    leg = ax.legend(loc='upper center', ncol=3, handlelength=1.4, columnspacing=1.0,
                    handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
                    edgecolor='gray', framealpha=0.9)
    leg.get_frame().set_linewidth(0.5)

    fig.tight_layout(pad=0.3)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 'q1_coherence_time.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

In [ ]:
n_bins = q2_t1_list.reshape(-1, BIN).shape[0]
x = np.linspace(0, Time, n_bins)

series = [
    (q2_t1_list, '$T_1$', 'C0'),
    (q2_t2_ram_list, '$T_2^*$', 'C1'),
    (q2_t2_list, '$T_2^E$', 'C2'),
]

with plt.rc_context(PRA):
    fig, ax = plt.subplots(figsize=(3.375, 2.1))
    for data, label, color in series:
        binned = data.reshape(-1, BIN)
        mean = np.nanmean(binned, axis=1)
        std = np.nanstd(binned, axis=1)
        ax.fill_between(x, mean - std, mean + std, alpha=0.25, color=color, linewidth=0)
        ax.plot(x, mean, color=color, label=label)

    ax.set_xlim(0, Time)
    ax.set_ylim(0, 100)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Q2 coherence time (μs)')
    leg = ax.legend(loc='upper center', ncol=3, handlelength=1.4, columnspacing=1.0,
                    handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
                    edgecolor='gray', framealpha=0.9)
    leg.get_frame().set_linewidth(0.5)

    fig.tight_layout(pad=0.3)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 'q2_coherence_time.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

### 3.2 over current

In [ ]:
multi_t1_and_t2_tuid_list = np.genfromtxt(get_datadir() + '/multi_t1t2_curr_0809_tuid_arr.txt', dtype='str')
current = np.unique([dataset_current(tuid) for tuid in multi_t1_and_t2_tuid_list[:50]])

In [ ]:
t1_t2_curr = []

for tuid_idx, tuid in enumerate(tqdm(multi_t1_and_t2_tuid_list)):
    t1_t2_curr.append(t1_and_t2_fit(load_dataset(tuid), r_squared=0.6))

t1_t2_curr = np.array(t1_t2_curr)

In [ ]:
[N_run, _, _, N_rep] = t1_t2_curr.shape
N_curr = current.size
N_run = N_run // N_curr

q1_t1_mean_curr = []
q1_t1_median_curr = []
q1_t1_std_curr = []
q1_t2_mean_curr = []
q1_t2_median_curr = []
q1_t2_std_curr = []
q2_t1_mean_curr = []
q2_t1_median_curr = []
q2_t1_std_curr = []
q2_t2_mean_curr = []
q2_t2_median_curr = []
q2_t2_std_curr = []

for i in range(N_curr):
    q1_t1_mean_curr.append(np.nanmean(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 0, 0]))
    q1_t1_median_curr.append(np.nanmedian(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 0, 0]))
    q1_t1_std_curr.append(np.nanstd(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 0, 0]))
    q1_t2_mean_curr.append(np.nanmean(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 0, 1]))
    q1_t2_median_curr.append(np.nanmedian(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 0, 1]))
    q1_t2_std_curr.append(np.nanstd(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 0, 1]))
    q2_t1_mean_curr.append(np.nanmean(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 1, 0]))
    q2_t1_median_curr.append(np.nanmedian(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 1, 0]))
    q2_t1_std_curr.append(np.nanstd(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 1, 0]))
    q2_t2_mean_curr.append(np.nanmean(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 1, 1]))
    q2_t2_median_curr.append(np.nanmedian(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 1, 1]))
    q2_t2_std_curr.append(np.nanstd(t1_t2_curr[np.r_[[i + N_curr * n for n in range(N_run)]], 1, 1]))

q1_t1_mean_curr = np.array(q1_t1_mean_curr)
q1_t1_median_curr = np.array(q1_t1_median_curr)
q1_t1_std_curr = np.array(q1_t1_std_curr)
q1_t2_mean_curr = np.array(q1_t2_mean_curr)
q1_t2_median_curr = np.array(q1_t2_median_curr)
q1_t2_std_curr = np.array(q1_t2_std_curr)
q2_t1_mean_curr = np.array(q2_t1_mean_curr)
q2_t1_median_curr = np.array(q2_t1_median_curr)
q2_t1_std_curr = np.array(q2_t1_std_curr)
q2_t2_mean_curr = np.array(q2_t2_mean_curr)
q2_t2_median_curr = np.array(q2_t2_median_curr)
q2_t2_std_curr = np.array(q2_t2_std_curr)

In [ ]:
q1_tphi_median_curr = (1 / (q1_t2_median_curr) - 1 / (2 * q1_t1_median_curr)) ** -1
sigma_gamma_phi = np.sqrt((q1_t2_std_curr / q1_t2_mean_curr**2)**2 + (q1_t1_std_curr / (2 * q1_t1_mean_curr**2))**2)
q1_tphi_std_curr = q1_tphi_median_curr**2 * sigma_gamma_phi

q2_tphi_median_curr = (1 / (q2_t2_median_curr) - 1 / (2 * q2_t1_median_curr)) ** -1
sigma_gamma_phi = np.sqrt((q2_t2_std_curr / q2_t2_mean_curr**2)**2 + (q2_t1_std_curr / (2 * q2_t1_mean_curr**2))**2)
q2_tphi_std_curr = q2_tphi_median_curr**2 * sigma_gamma_phi

In [ ]:
with plt.rc_context(PRA):
    fig, ax = plt.subplots(figsize=(3.375, 2.1))
    ax.errorbar(curr2flux(current), q1_t1_mean_curr, yerr=q1_t1_std_curr, c='C0', capsize=3,
               elinewidth=0.7, ms=3.5, fmt='o', label='$T_1$')
    ax.errorbar(curr2flux(current), q1_t2_mean_curr, yerr=q1_t2_std_curr, c='C2', capsize=3,
               elinewidth=0.7, ms=3.5, fmt='o', label='$T_2^E$')

    ax.set_xlabel(r'SNAIL flux ($\Phi/\Phi_0$)')
    ax.set_ylabel('Q1 coherence time (μs)')
    ax.set_ylim(0, 100)
    leg = ax.legend(loc='upper right', ncol=1, handlelength=1.4, columnspacing=1.0,
                    handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
                    edgecolor='gray', framealpha=0.9)
    leg.get_frame().set_linewidth(0.5)

    fig.tight_layout(pad=0.3)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 'q1_coherence_vs_flux.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

In [ ]:
with plt.rc_context(PRA):
    fig, ax = plt.subplots(figsize=(3.375, 2.1))
    ax.plot(curr2flux(current), q1_tphi_median_curr, 'o', c='C3', label=r'$T_\varphi$')
    # ax.errorbar(curr2flux(current), q1_tphi_median_curr, yerr=q1_tphi_std_curr, c='C3',
    #             capsize=3, elinewidth=0.7, ms=3.5, fmt='o', label=r'$T_\varphi$')

    ax.set_xlabel(r'SNAIL flux ($\Phi/\Phi_0$)')
    ax.set_ylabel(r'Q1 $T_\varphi$ (μs)')
    ax.set_ylim(0, 400)
    # leg = ax.legend(loc='upper right', ncol=1, handlelength=1.4, columnspacing=1.0,
    #                 handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
    #                 edgecolor='gray', framealpha=0.9)
    leg.get_frame().set_linewidth(0.5)

    fig.tight_layout(pad=0.3)
    # for ext in ('pdf', 'png'):
    #     fig.savefig(os.path.join(FIGDIR, 'q1_tphi_vs_flux.%s' % ext),
    #                 bbox_inches='tight', pad_inches=0.02)
    plt.show()

In [ ]:
with plt.rc_context(PRA):
    fig, ax = plt.subplots(figsize=(3.375, 2.1))
    ax.errorbar(curr2flux(current), q2_t1_mean_curr, yerr=q2_t1_std_curr, c='C0', capsize=3,
               elinewidth=0.7, ms=3.5, fmt='o', label='$T_1$')
    ax.errorbar(curr2flux(current), q2_t2_mean_curr, yerr=q2_t2_std_curr, c='C2', capsize=3,
               elinewidth=0.7, ms=3.5, fmt='o', label='$T_2^E$')

    ax.set_xlabel(r'SNAIL flux ($\Phi/\Phi_0$)')
    ax.set_ylabel('Q2 coherence time (μs)')
    ax.set_ylim(0, 100)
    leg = ax.legend(loc='upper right', ncol=1, handlelength=1.4, columnspacing=1.0,
                    handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
                    edgecolor='gray', framealpha=0.9)
    leg.get_frame().set_linewidth(0.5)

    fig.tight_layout(pad=0.3)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 'q2_coherence_vs_flux.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

In [ ]:
with plt.rc_context(PRA):
    fig, ax = plt.subplots(figsize=(3.375*2, 2.1*2))
    ax.plot(curr2flux(current), q1_tphi_median_curr, 'o', ms=3.5, c='C0', label='Q1')
    ax.plot(curr2flux(current), q2_tphi_median_curr, 'o', ms=3.5, c='C1', label='Q2')
    # ax.errorbar(curr2flux(current), q2_tphi_median_curr, yerr=q2_tphi_std_curr, c='C3',
    #             capsize=3, elinewidth=0.7, ms=3.5, fmt='o', label=r'$T_\varphi$')

    ax.set_xlabel(r'SNAIL flux ($\Phi/\Phi_0$)')
    ax.set_ylabel(r'$T_\varphi$ (μs)')
    ax.set_ylim(0, 200)
    ax.set_xlim(0.39, 0.42)
    leg = ax.legend(loc='upper right', ncol=1, handlelength=1.4, columnspacing=1.0,
                    handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
                    edgecolor='gray', framealpha=0.9)
    leg.get_frame().set_linewidth(0.5)

    fig.tight_layout(pad=0.3)
    # for ext in ('pdf', 'png'):
    #     fig.savefig(os.path.join(FIGDIR, 'q2_tphi_vs_flux.%s' % ext),
    #                 bbox_inches='tight', pad_inches=0.02)
    plt.show()

### 3.3 Combined

In [ ]:
# Combined 2x2 figure, PRA two-column width (6.75 in -- see cell 2's note on the
# single- to two-column conversion). Top row is coherence time vs SNAIL flux, bottom row
# is coherence time vs measurement time -- columns are Q1 (left) and Q2 (right). Panels
# are not sharex since the top and bottom rows have unrelated x quantities (flux vs hours).
# Series styling (colors, markers, labels) is copied verbatim from cells 10, 11, 17, 18 --
# the flux rows use C0/C2 for T1/T2E (no T2* there) and marker 'o' for both qubits.
# Each of the four panels carries its own legend. wspace/hspace are widened from the
# tight_layout default so the panels and their legends don't crowd each other.
with plt.rc_context(PRA):
    fig, axes = plt.subplots(2, 2, figsize=(6.75, 4),
                             gridspec_kw=dict(wspace=0.25, hspace=0.35))

    flux_series = [
        (axes[0, 0], q1_t1_mean_curr, q1_t1_std_curr, q1_t2_mean_curr, q1_t2_std_curr, 'Q1', '(a)'),
        (axes[0, 1], q2_t1_mean_curr, q2_t1_std_curr, q2_t2_mean_curr, q2_t2_std_curr, 'Q2', '(b)'),
    ]
    for ax, t1_mean, t1_std, t2_mean, t2_std, qlabel, panel in flux_series:
        ax.errorbar(curr2flux(current), t1_mean, yerr=t1_std, c='C0', capsize=3,
                    elinewidth=0.7, ms=3.5, fmt='o', label='$T_1$')
        ax.errorbar(curr2flux(current), t2_mean, yerr=t2_std, c='C2', capsize=3,
                    elinewidth=0.7, ms=3.5, fmt='o', label='$T_2^E$')
        ax.set_ylim(0, 100)
        ax.set_xlabel(r'SNAIL flux ($\Phi/\Phi_0$)')
        ax.set_ylabel(qlabel + ' coherence time (μs)')
        leg = ax.legend(loc='upper right', ncol=1, handlelength=1.4, columnspacing=1.0,
                        handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
                        edgecolor='gray', framealpha=0.9)
        leg.get_frame().set_linewidth(0.5)
        ax.text(-0.2, 1.15, panel, transform=ax.transAxes, ha='left', va='top')

    time_series = [
        (axes[1, 0], q1_t1_list, q1_t2_ram_list, q1_t2_list, 'Q1', '(c)'),
        (axes[1, 1], q2_t1_list, q2_t2_ram_list, q2_t2_list, 'Q2', '(d)'),
    ]
    for ax, t1_data, t2star_data, t2e_data, qlabel, panel in time_series:
        n_bins = t1_data.reshape(-1, BIN).shape[0]
        x = np.linspace(0, Time, n_bins)
        for data, label, color in [(t1_data, '$T_1$', 'C0'),
                                    (t2star_data, '$T_2^*$', 'C1'),
                                    (t2e_data, '$T_2^E$', 'C2')]:
            binned = data.reshape(-1, BIN)
            mean = np.nanmean(binned, axis=1)
            std = np.nanstd(binned, axis=1)
            ax.fill_between(x, mean - std, mean + std, alpha=0.25, color=color, linewidth=0)
            ax.plot(x, mean, color=color, label=label)
        ax.set_xlim(0, Time)
        ax.set_ylim(0, 100)
        ax.set_xlabel('Time (hours)')
        ax.set_ylabel(qlabel + ' coherence time (μs)')
        leg = ax.legend(loc='upper center', ncol=3, handlelength=1.4, columnspacing=1.0,
                        handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
                        edgecolor='gray', framealpha=0.9)
        leg.get_frame().set_linewidth(0.5)
        ax.text(-0.2, 1.15, panel, transform=ax.transAxes, ha='left', va='top')

    fig.tight_layout(pad=0.3)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 'coherence_combined.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

## 4. Gate Tuneup

In [ ]:
swap_list_tuid = np.genfromtxt(get_datadir() + '/iSWAP_pump_curr_9dBm_4.txt', dtype='str')
current = np.unique([dataset_current(tuid) for tuid in swap_list_tuid[:20]])

### 4.1 over current

Every repeat of every dataset is fitted separately and the results averaged, so the spread over the
repeats is a real error bar. Traces whose oscillation is not resolved (fewer than 4 samples per
period, or a fitted contrast under 5% of the range) and traces whose `offset_tau` runs past what a
30 μs window can constrain are dropped before averaging.

In [ ]:
from snl_qblox.analyzer import swap_metrics

N_COL = 7
T_NS, TAU_OSC, OFF_TAU, SWAP_ERR, ETA, N_SWAP, R_SQ = range(N_COL)

swap_fit = []
swap_fit_std = []
n_dropped = 0

for swap_tuid in tqdm(swap_list_tuid):
    swap_ds = load_dataset(swap_tuid)
    x0_data = np.unique(swap_ds.x0)
    traces = swap_ds.y0.values.reshape(-1, x0_data.size)      # the repeats, kept separate

    per_trace = []
    for y0_data in traces:
        guess = swap_decay_model.guess(y0_data, t=x0_data)
        fit_result = swap_decay_model.fit(y0_data, params=guess, t=x0_data)
        metrics = swap_metrics(fit_result, x0_data)

        runaway = metrics['offset_tau'] > 5 * (x0_data[-1] - x0_data[0])
        if not metrics['resolved'] or runaway:
            n_dropped += 1
            continue

        t_iswap = metrics['iswap_time']
        k = int(np.argmin(np.abs(x0_data - t_iswap)))
        per_trace.append((
            t_iswap * 1e9,                                             # gate time [ns]
            metrics['tau_osc'] * 1e6,                                  # exchange contrast decay [us]
            metrics['offset_tau'] * 1e6,                               # excitation lifetime [us]
            100 * metrics['bottom_start'] / metrics['contrast_start'],  # swap error bound [%]
            np.exp(-t_iswap / metrics['offset_tau']) - metrics['bottom'][k],   # transfer efficiency
            metrics['tau_osc'] / t_iswap,                              # swaps per coherence time
            fit_result.rsquared,
        ))

    per_trace = np.array(per_trace)
    swap_fit.append(per_trace.mean(axis=0))
    swap_fit_std.append(per_trace.std(axis=0))

swap_fit = np.array(swap_fit)
swap_fit_std = np.array(swap_fit_std)

print(f'{len(swap_list_tuid)} datasets x {traces.shape[0]} repeats = '
      f'{len(swap_list_tuid) * traces.shape[0]} fits, {n_dropped} traces dropped')
print(f'R^2 median {np.median(swap_fit[:, R_SQ]):.4f}, worst {swap_fit[:, R_SQ].min():.4f}')

In [ ]:
# cell 31 takes only swap_list_tuid[:20], one short of a full ramp, so rebuild the
# current axis from every dataset and group by it rather than assuming the ordering
swap_current = np.array([dataset_current(swap_tuid) for swap_tuid in swap_list_tuid])
current = np.unique(swap_current)

swap_mean_curr = np.array([swap_fit[swap_current == c].mean(axis=0) for c in current])
swap_std_curr = np.array([swap_fit[swap_current == c].std(axis=0) for c in current])

print(f'{current.size} currents, {swap_fit.shape[0] // current.size} runs each')

best = int(np.argmax(swap_mean_curr[:, N_SWAP]))
print(f'best operating point: {current[best] * 1e3:.0f} uA '
      f'({curr2flux(current[best]):.4f} flux quanta)')
print(f'  t_iSWAP        {swap_mean_curr[best, T_NS]:.0f} +- {swap_std_curr[best, T_NS]:.0f} ns')
print(f'  tau_osc        {swap_mean_curr[best, TAU_OSC]:.1f} +- {swap_std_curr[best, TAU_OSC]:.1f} us')
print(f'  offset_tau     {swap_mean_curr[best, OFF_TAU]:.1f} +- {swap_std_curr[best, OFF_TAU]:.1f} us')
print(f'  swaps per tau  {swap_mean_curr[best, N_SWAP]:.1f}')
print(f'  transfer eff   {swap_mean_curr[best, ETA]:.4f}')
print(f'  swap error     <= {swap_mean_curr[best, SWAP_ERR]:.1f} %')
print(f'  coherence-limited gate error ~ {0.8 / swap_mean_curr[best, N_SWAP]:.4f} '
      f'(F_avg ~ {100 * (1 - 0.8 / swap_mean_curr[best, N_SWAP]):.2f} %)')

In [ ]:
flux = curr2flux(current)

# t_swap / tau_swap is the inverse of the stored swaps-per-coherence-time column,
# so its uncertainty follows from that column's spread over the runs
ratio = 1 / swap_mean_curr[:, N_SWAP]
ratio_err = swap_std_curr[:, N_SWAP] / swap_mean_curr[:, N_SWAP] ** 2

with plt.rc_context(PRA):
    fig, axes = plt.subplots(2, 1, figsize=(3.375, 3.6), sharex=True)

    ax_t = axes[0]
    ax_tau = ax_t.twinx()
    ax_t.tick_params(axis='y', which='both', right=False, colors='C0')

    ax_t.plot(flux, swap_mean_curr[:, T_NS], 'o', ms=3.5, c='C0')
    ax_t.set_ylabel(r'$t_\mathrm{swap}$ (ns)', c='C0')

    ax_tau.errorbar(flux, swap_mean_curr[:, TAU_OSC], yerr=swap_std_curr[:, TAU_OSC],
                    c='C1', capsize=3, elinewidth=0.7, ms=3.5, fmt='s',
                    label=r'$\tau_\mathrm{swap}$')
    ax_tau.errorbar(flux, swap_mean_curr[:, OFF_TAU], yerr=swap_std_curr[:, OFF_TAU],
                    c='C2', capsize=3, elinewidth=0.7, ms=3.5, fmt='^',
                    label=r'$\tau_\mathrm{offset}$')
    ax_tau.set_ylabel('Coherence (μs)')
    ax_tau.set_ylim(0, None)
    leg = ax_tau.legend(loc='lower left', ncol=1, handlelength=1.2, borderpad=0.3,
                        handletextpad=0.4, frameon=True, facecolor='white',
                        edgecolor='gray', framealpha=0.9)
    leg.get_frame().set_linewidth(0.5)

    axes[1].errorbar(flux, ratio, yerr=ratio_err, c='C4', capsize=3,
                     elinewidth=0.7, ms=3.5, fmt='o')
    axes[1].set_ylabel(r'$t_\mathrm{swap}/\tau_\mathrm{swap}$')
    axes[1].set_xlabel(r'SNAIL flux ($\Phi/\Phi_0$)')

    for ax in (ax_t, axes[1]):
        ax.margins(x=0.04)

    fig.tight_layout(pad=0.3)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 'iswap_vs_current.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

### 4.1.1 T1 and T2 over current

Two sweeps of the same 21 currents: **18 Sept** (210 datasets, 10 ramps) and **19–21 Sept**
(420 datasets, 20 ramps). Every repeat of every dataset is fitted separately and then pooled
per current across both runs.

Three guards run before anything is averaged, all judged against each run's own statistics rather
than fixed numbers, so a repeat is rejected when it is more than 5 median-absolute-deviations
below that run's median in:

* **preparation** — the first point of a T1 trace, which should be $1 - e_1$ when the $\pi$ pulse
  lands (echo traces start in $|0\rangle$, so this applies to case 1 only);
* **contrast** — the peak-to-peak of the trace, which catches degraded readout;
* **fit quality** — $R^2$, judged per case, since an echo trace is intrinsically noisier than a
  T1 trace and a single threshold would treat them unequally.

The figure pools the two runs: every surviving repeat at a given current counts once, so the
error bars carry the run-to-run difference as well as the repeat scatter. The ratio printed below
is the check that pooling is fair.

In [ ]:
from scipy.optimize import curve_fit


def fit_decay(t, y):
    """Fit y = a*exp(-t/tau) + c and return (tau_us, r_squared).

    curve_fit with a seed read straight off the data, rather than lmfit: same answer to
    five decimals, about 20x faster, and the fitting is what dominates this loop.
    """
    try:
        popt, _ = curve_fit(lambda tt, a, tau, c: a * np.exp(-tt / tau) + c,
                            t, y, p0=[y[0] - y[-1], t[-1] / 3, y[-1]], maxfev=2000)
    except Exception:
        return np.nan, 0.0
    residual = y - (popt[0] * np.exp(-t / popt[1]) + popt[2])
    centred = y - y.mean()
    return popt[1] * 1e6, 1 - residual.dot(residual) / centred.dot(centred)


RUNS = ("0918", "0919")

records = []
for run, tag in enumerate(RUNS):
    tuids = np.genfromtxt(get_datadir() + '/multi_t1t2_curr075_%s_tuid_arr.txt' % tag, dtype='str')
    print("%s: %d datasets" % (tag, len(tuids)))
    for tuid in tqdm(tuids, desc=tag):
        ds = load_dataset(tuid)
        delay = np.unique(ds.x0)
        curr = dataset_current(tuid)
        for case in (1, 2):                      # 1 = T1, 2 = echo; case 3 ignored
            sub = ds.where(ds.x1 == case, drop=True)
            for qubit, var in enumerate(("y0", "y1")):
                for y in sub[var].values.reshape(-1, delay.size):
                    if np.isnan(y).any():        # a dataset still being written
                        continue
                    tau_us, r_squared = fit_decay(delay, y)
                    records.append((run, curr, case, qubit, y[0], np.ptp(y), tau_us, r_squared))

records = np.array(records)
RUN, CURR, CASE, QUBIT, FIRST, CONTRAST, TAU, R_SQ = range(8)
print("%d repeats fitted over both runs" % len(records))

In [ ]:
def far_below(values, n_mad=5):
    """True where a value sits more than n_mad median-absolute-deviations below the median."""
    median = np.median(values)
    mad = np.median(np.abs(values - median))
    return values < median - n_mad * mad


keep = np.ones(len(records), bool)
for run in range(len(RUNS)):
    for case in (1, 2):
        for qubit in (0, 1):
            group = np.flatnonzero((records[:, RUN] == run) & (records[:, CASE] == case)
                                   & (records[:, QUBIT] == qubit))
            bad = far_below(records[group, CONTRAST]) | far_below(records[group, R_SQ])
            if case == 1:                        # only a T1 trace starts excited
                bad |= far_below(records[group, FIRST])
            keep[group[bad]] = False

print("%-22s %9s %9s %10s %9s" % ("", "T1 [us]", "T2E [us]", "T_phi [us]", "rejected"))
for run, tag in enumerate(RUNS):
    for qubit in (0, 1):
        sel = (records[:, RUN] == run) & (records[:, QUBIT] == qubit)
        t1 = np.median(records[sel & (records[:, CASE] == 1) & keep, TAU])
        t2 = np.median(records[sel & (records[:, CASE] == 2) & keep, TAU])
        print("%-22s %9.1f %9.1f %10.1f %8.1f%%" % (
            "%s  Q%d" % (tag, qubit + 1), t1, t2, 1 / (1 / t2 - 1 / (2 * t1)),
            100 * (sel & ~keep).sum() / sel.sum()))

In [ ]:
current = np.unique(records[:, CURR])


def per_current(case, qubit, run=None):
    """Median over the surviving repeats at each current, and the spread between them.

    With `run` given the result is for that run alone; otherwise both runs are pooled,
    so the spread includes the run-to-run difference as well as the repeat scatter.
    """
    sel = (records[:, CASE] == case) & (records[:, QUBIT] == qubit) & keep
    if run is not None:
        sel = sel & (records[:, RUN] == run)
    median, spread = [], []
    for c in current:
        at_c = records[sel & (np.abs(records[:, CURR] - c) < 1e-9), TAU]
        median.append(np.median(at_c))
        spread.append(np.diff(np.percentile(at_c, [16, 84]))[0] / 2)
    return np.array(median), np.array(spread)


# both runs pooled - this is what the figure shows
t1_curr, t1_err, t2_curr, t2_err, tphi_curr = {}, {}, {}, {}, {}
for qubit in (0, 1):
    t1_curr[qubit], t1_err[qubit] = per_current(1, qubit)
    t2_curr[qubit], t2_err[qubit] = per_current(2, qubit)
    tphi_curr[qubit] = 1 / (1 / t2_curr[qubit] - 1 / (2 * t1_curr[qubit]))

print("pooled over both runs:")
for qubit in (0, 1):
    print("  Q%d: T1 %.1f-%.1f us | T2E %.1f-%.1f us | T_phi %.1f-%.1f us | T2E/2T1 %.2f-%.2f"
          % (qubit + 1, t1_curr[qubit].min(), t1_curr[qubit].max(),
             t2_curr[qubit].min(), t2_curr[qubit].max(),
             tphi_curr[qubit].min(), tphi_curr[qubit].max(),
             (t2_curr[qubit] / (2 * t1_curr[qubit])).min(),
             (t2_curr[qubit] / (2 * t1_curr[qubit])).max()))

# the runs are pooled only because they agree; this is the check that they do
print()
print("run-to-run ratio, %s / %s, over the 21 currents:" % (RUNS[1], RUNS[0]))
for qubit in (0, 1):
    for name, case in (("T1", 1), ("T2E", 2)):
        first, _ = per_current(case, qubit, run=0)
        second, _ = per_current(case, qubit, run=1)
        ratio = second / first
        print("  Q%d %-4s median %.3f  (5-95%% %.3f - %.3f)" % (
            qubit + 1, name, np.median(ratio), np.percentile(ratio, 5), np.percentile(ratio, 95)))

In [ ]:
flux = curr2flux(current)

with plt.rc_context(PRA):
    fig, axes = plt.subplots(3, 1, figsize=(3.375, 5.6), sharex=True)

    # colour carries the quantity in the top two panels, the qubit in the bottom one
    for ax, qubit in ((axes[0], 0), (axes[1], 1)):
        ax.errorbar(flux, t1_curr[qubit], yerr=t1_err[qubit], c='C0', capsize=2,
                    elinewidth=0.6, ms=3.2, fmt='o', label='$T_1$')
        ax.errorbar(flux, t2_curr[qubit], yerr=t2_err[qubit], c='C2', capsize=2,
                    elinewidth=0.6, ms=3.2, fmt='s', label='$T_2^E$')
        ax.set_ylabel('Q%d time (μs)' % (qubit + 1))
        ax.set_ylim(25, 70)
        leg = ax.legend(loc='lower left', ncol=2, handlelength=1.2, columnspacing=0.8,
                        handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
                        edgecolor='gray', framealpha=0.9)
        leg.get_frame().set_linewidth(0.5)

    axes[2].plot(flux, tphi_curr[0], 'o-', c='C3', ms=3.2, lw=0.8, label='Q1')
    axes[2].plot(flux, tphi_curr[1], '^--', c='C3', ms=3.2, lw=0.8, mfc='white', label='Q2')
    axes[2].set_ylabel(r'$T_\varphi$ (μs)')
    axes[2].set_xlabel(r'SNAIL flux ($\Phi/\Phi_0$)')
    leg = axes[2].legend(loc='lower left', ncol=2, handlelength=1.4, columnspacing=0.8,
                         handletextpad=0.4, borderpad=0.3, frameon=True, facecolor='white',
                         edgecolor='gray', framealpha=0.9)
    leg.get_frame().set_linewidth(0.5)

    for ax in axes:
        ax.margins(x=0.04)

    fig.tight_layout(pad=0.3)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 't1_t2_vs_current.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

### 4.2 over power

The 50-pass power sweep of 21 Sept: each dataset is one pass holding all 21 pump settings, frequency
and amplitude advanced together, with the readout recalibrated every five passes. Each setting in
each pass is fitted separately and the results averaged over passes, so the spread is a real error
bar. The guards are the same as in 4.1.

One difference from 4.1: `offset_f` is pinned to the acceptor's reading at zero swap duration (the
readout floor $e_0$) rather than fitted. Left free it is 99% correlated with `offset_tau`, and on
this sweep's 20-50 μs windows the free fit drifts `offset_tau` by tens of percent.

In [ ]:
pump_power_list = W2dBm(np.linspace(0.0001, 0.1, 41))

pump_freq_list = np.array([
    2.57932710e+08, 2.58146482e+08, 2.58360254e+08, 2.58574026e+08,
    2.58787797e+08, 2.59001569e+08, 2.59215341e+08, 2.59429113e+08,
    2.59642885e+08, 2.59856656e+08, 2.60070428e+08, 2.60284200e+08,
    2.60497972e+08, 2.60711744e+08, 2.60925516e+08, 2.61139287e+08,
    2.61353059e+08, 2.61566831e+08, 2.61780603e+08, 2.61994375e+08,
    2.62208147e+08, 2.62421918e+08, 2.62635690e+08, 2.62849462e+08,
    2.63063234e+08, 2.63277006e+08, 2.63490777e+08, 2.63704549e+08,
    2.63918321e+08, 2.64132093e+08, 2.64345865e+08, 2.64559637e+08,
    2.64773408e+08, 2.64987180e+08, 2.65200952e+08, 2.65414724e+08,
    2.65628496e+08, 2.65842267e+08, 2.66056039e+08, 2.66269811e+08,
    2.66483583e+08
])

t_swap_list = np.array([
    4796.87951867,  872.41669099,  613.21499262,  497.68689662,
     428.86984098,  382.00017639,  347.47954774,  320.70919875,
     299.17544418,  281.37382545,  266.34193466,  253.43187024,
     242.18945383,  232.28566746,  223.47557665,  215.57260429,
     208.43180823,  201.93866232,  196.00132168,  190.5451622 ,
     185.50884427,  180.84142245,  176.50018783,  172.44903332,
     168.6571985 ,  165.0982941 ,  161.74953546,  158.59113394,
     155.60580932,  152.77839576,  150.09552079,  147.54534186,
     145.11732873,  142.8020824 ,  140.59118375,  138.47706615,
     136.4529077 ,  134.51253971,  132.65036853,  130.86130849,
     129.14072413
]) / 1e9

In [ ]:
pump_power_list = Vp2dBm(np.linspace(dBm2Vp(-10), dBm2Vp(20), 41))

pump_freq_list = np.array([
    2.57932710e+08, 2.57950832e+08, 2.57978988e+08, 2.58017177e+08,
    2.58065399e+08, 2.58123655e+08, 2.58191944e+08, 2.58270266e+08,
    2.58358621e+08, 2.58457010e+08, 2.58565432e+08, 2.58683888e+08,
    2.58812376e+08, 2.58950898e+08, 2.59099454e+08, 2.59258042e+08,
    2.59426664e+08, 2.59605320e+08, 2.59794008e+08, 2.59992730e+08,
    2.60201485e+08, 2.60420274e+08, 2.60649095e+08, 2.60887950e+08,
    2.61136839e+08, 2.61395761e+08, 2.61664716e+08, 2.61943704e+08,
    2.62232725e+08, 2.62531780e+08, 2.62840869e+08, 2.63159990e+08,
    2.63489145e+08, 2.63828333e+08, 2.64177555e+08, 2.64536809e+08,
    2.64906098e+08, 2.65285419e+08, 2.65674774e+08, 2.66074162e+08,
    2.66483583e+08
])

t_swap_list = np.array([
    4796.87951867, 2645.87856169, 1814.88664583, 1376.37655521,
    1106.17115898,  923.30332782,  791.48391463,  692.04675094,
     614.41957146,  552.17084099,  501.16576153,  458.62638065,
     422.61797066,  391.75210186,  365.00681874,  341.61339395,
     320.9825915 ,  302.65525216,  286.26833512,  271.53105916,
     258.20780692,  246.1056582 ,  235.06515426,  224.9533573 ,
     215.65856673,  207.08624873,  199.15586643,  191.79838658,
     184.9543002 ,  178.5720379 ,  172.60669096,  167.01897166,
     161.77436218,  156.84241343,  152.19616384,  147.81165496,
     143.66752542,  139.74466905,  136.02594545,  132.49593406,
     129.14072413
]) / 1e9

In [ ]:
power_W = np.array([0.001, 0.00245, 0.0039, 0.00535, 0.0068, 0.00825, 0.0097,
                    0.01115, 0.0126, 0.01405, 0.0155, 0.01695, 0.0184, 0.01985,
                    0.0213, 0.02275, 0.0242, 0.02565, 0.0271, 0.02855, 0.03])
P_mW = 1e3 * power_W

f_opt = 1e-9 * np.array([2.57927786e8, 2.58048016e8, 2.58206992e8, 2.58342773e8,
                         2.58485067e8, 2.58628026e8, 2.58746599e8, 2.58959399e8,
                         2.59113346e8, 2.59239451e8, 2.59398258e8, 2.59451992e8,
                         2.59637337e8, 2.59704222e8, 2.59856924e8, 2.59814956e8,
                         2.60125545e8, 2.60080634e8, 2.60370865e8, 2.60522089e8,
                         2.60644210e8])

t_gate = np.array([1430.89534908, 899.04974534, 714.4102194, 604.20519246,
                   533.35797234, 481.44829084, 442.88699757, 415.38642205,
                   387.11145324, 364.26370528, 344.07038045, 326.32759552,
                   313.8624228, 297.52544188, 285.64785854, 275.9350227,
                   262.39357591, 255.54257086, 247.16073838, 239.15164503,
                   231.68860879])   

In [ ]:
from snl_qblox.analyzer import swap_metrics

N_COL = 7                                                   # same columns as 4.1
T_NS, TAU_OSC, OFF_TAU, SWAP_ERR, ETA, N_SWAP, R_SQ = range(N_COL)

POWER_RUN = ('20260921-163336', '20260921-202251')     # first and last pass of the sweep

day_dir = os.path.join(get_datadir(), POWER_RUN[0][:8])
power_list_tuid = [d[:26] for d in sorted(os.listdir(day_dir))
                   if d.endswith('SNAIL iSWAP') and POWER_RUN[0] <= d[:15] <= POWER_RUN[1]]
np.savetxt(get_datadir() + '/iSWAP_pump_power_0921_tuid_arr.txt', power_list_tuid, fmt='%s')


def split_blocks(t):
    """Index ranges of the per-setting blocks: the duration axis restarts at each one."""
    starts = np.flatnonzero(np.r_[True, np.diff(t) < 0])
    return list(zip(starts, np.r_[starts[1:], t.size]))


N_POWER = power_W.size
power_fit = np.full((len(power_list_tuid), N_POWER, N_COL), np.nan)   # pass, setting, column
n_dropped_p = 0

for j, swap_tuid in enumerate(tqdm(power_list_tuid)):
    swap_ds = load_dataset(swap_tuid)
    t_all, y0_all, y1_all = swap_ds.x0.values, swap_ds.y0.values, swap_ds.y1.values

    for i, (a, b) in enumerate(split_blocks(t_all)):
        x0_data, y0_data = t_all[a:b], y0_all[a:b]
        guess = swap_decay_model.guess(y0_data, t=x0_data)
        guess['offset_f'].set(value=y1_all[a], vary=False)       # pin to the readout floor
        fit_result = swap_decay_model.fit(y0_data, params=guess, t=x0_data)
        metrics = swap_metrics(fit_result, x0_data)

        runaway = metrics['offset_tau'] > 5 * (x0_data[-1] - x0_data[0])
        if not metrics['resolved'] or runaway:
            n_dropped_p += 1
            continue

        t_iswap = metrics['iswap_time']
        k = int(np.argmin(np.abs(x0_data - t_iswap)))
        power_fit[j, i] = (
            t_iswap * 1e9,                                             # gate time [ns]
            metrics['tau_osc'] * 1e6,                                  # exchange contrast decay [us]
            metrics['offset_tau'] * 1e6,                               # excitation lifetime [us]
            100 * metrics['bottom_start'] / metrics['contrast_start'],  # swap error bound [%]
            np.exp(-t_iswap / metrics['offset_tau']) - metrics['bottom'][k],   # transfer efficiency
            metrics['tau_osc'] / t_iswap,                              # swaps per coherence time
            fit_result.rsquared,
        )

print(f'{len(power_list_tuid)} passes x {N_POWER} settings = '
      f'{len(power_list_tuid) * N_POWER} fits, {n_dropped_p} dropped')
print(f'R^2 median {np.nanmedian(power_fit[:, :, R_SQ]):.4f}, '
      f'worst {np.nanmin(power_fit[:, :, R_SQ]):.4f}')

In [ ]:
swap_mean_pow = np.nanmean(power_fit, axis=0)
swap_std_pow = np.nanstd(power_fit, axis=0)

best_p = int(np.argmax(swap_mean_pow[:, N_SWAP]))
print(f'{N_POWER} pump settings, {power_fit.shape[0]} passes each')
print(f'best operating point: {P_mW[best_p]:.2f} mW ({10 * np.log10(P_mW[best_p]):.1f} dBm), '
      f'pump at {f_opt[best_p]:.5f} GHz')
print(f'  t_iSWAP        {swap_mean_pow[best_p, T_NS]:.0f} +- {swap_std_pow[best_p, T_NS]:.0f} ns')
print(f'  tau_osc        {swap_mean_pow[best_p, TAU_OSC]:.1f} +- {swap_std_pow[best_p, TAU_OSC]:.1f} us')
print(f'  offset_tau     {swap_mean_pow[best_p, OFF_TAU]:.1f} +- {swap_std_pow[best_p, OFF_TAU]:.1f} us')
print(f'  swaps per tau  {swap_mean_pow[best_p, N_SWAP]:.1f}')
print(f'  transfer eff   {swap_mean_pow[best_p, ETA]:.4f}')
print(f'  swap error     <= {swap_mean_pow[best_p, SWAP_ERR]:.1f} %')
print(f'  coherence-limited gate error ~ {0.8 / swap_mean_pow[best_p, N_SWAP]:.4f} '
      f'(F_avg ~ {100 * (1 - 0.8 / swap_mean_pow[best_p, N_SWAP]):.2f} %)')

In [ ]:
# t_swap / tau_swap is the inverse of the stored swaps-per-coherence-time column,
# so its uncertainty follows from that column's spread over the passes
ratio_p = 1 / swap_mean_pow[:, N_SWAP]
ratio_p_err = swap_std_pow[:, N_SWAP] / swap_mean_pow[:, N_SWAP] ** 2

with plt.rc_context(PRA):
    fig, axes = plt.subplots(2, 1, figsize=(3.375, 3.6), sharex=True)

    ax_t = axes[0]
    ax_tau = ax_t.twinx()
    ax_t.tick_params(axis='y', which='both', right=False, colors='C0')

    ax_t.plot(P_mW, swap_mean_pow[:, T_NS], 'o', ms=3.5, c='C0')
    ax_t.set_ylabel(r'$t_\mathrm{swap}$ (ns)', c='C0')

    ax_tau.errorbar(P_mW, swap_mean_pow[:, TAU_OSC], yerr=swap_std_pow[:, TAU_OSC],
                    c='C1', capsize=3, elinewidth=0.7, ms=3.5, fmt='s',
                    label=r'$\tau_\mathrm{swap}$')
    ax_tau.errorbar(P_mW, swap_mean_pow[:, OFF_TAU], yerr=swap_std_pow[:, OFF_TAU],
                    c='C2', capsize=3, elinewidth=0.7, ms=3.5, fmt='^',
                    label=r'$\tau_\mathrm{offset}$')
    ax_tau.set_ylabel('Coherence (μs)')
    ax_tau.set_ylim(0, None)
    leg = ax_tau.legend(loc='upper right', ncol=1, handlelength=1.2, borderpad=0.3,
                        handletextpad=0.4, frameon=True, facecolor='white',
                        edgecolor='gray', framealpha=0.9)
    leg.get_frame().set_linewidth(0.5)

    axes[1].errorbar(P_mW, ratio_p, yerr=ratio_p_err, c='C4', capsize=3,
                     elinewidth=0.7, ms=3.5, fmt='o')
    axes[1].set_ylabel(r'$t_\mathrm{swap}/\tau_\mathrm{swap}$')
    axes[1].set_xlabel('Pump power (mW)')

    for ax in (ax_t, axes[1]):
        ax.margins(x=0.04)

    fig.tight_layout(pad=0.3)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 'iswap_vs_power.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

## 5. Tomography

### 5.1 Repeated Bell State

$|\Psi^+\rangle = (|01\rangle + |10\rangle)/\sqrt2$ is prepared with an $X_\pi$ on Q1 followed by the
$\sqrt{i\mathrm{SWAP}}$ and measured by two-qubit state tomography (9 settings) about 50 times in a row. Before
every run the loop measures a fresh readout confusion matrix (CM), saved next to the run's $\rho$ in
`bell_state/<set>_confusion.csv`. The processing is in `bell_state/bell_analysis.py`.

**Processing (pooled estimate).**
1. Each run's raw joint probabilities are corrected with *that run's* CM and linearly inverted,
   $\rho_{\rm lin} = \tfrac14\big(I + \sum_P \langle P\rangle P\big)$ — identical to `calculate_density_matrix`.
2. The $\rho_{\rm lin}$ of all runs are averaged. Everything up to here is linear, so this equals one
   tomography with all the shots.
3. One MLE (nearest physical state) is applied to the average, and $F = \langle\Psi^+|\rho|\Psi^+\rangle$.
4. The error bar is a bootstrap over runs (resample the runs, redo 2–3).

Averaging the per-run MLE fidelities instead (the previous method) is biased low: shot noise gives every
single run a negative eigenvalue, the MLE removes it at the cost of the Bell eigenvalue, and that loss has
one sign so it never averages out (≈ −0.7 % at 4000 shots, −1.6 % at 1000). It is printed for reference only.

**What the corrected fidelity means.** The CM is built from basis states prepared with $X_\pi$ pulses, with
the same idle before readout as a tomography rotation, so the correction removes readout error *and* the
decay during the tomography segment: the corrected $\rho$ is effectively the state right after the
$\sqrt{i\mathrm{SWAP}}$. It is therefore compared with the *simulated state* at that point. The raw
(uncorrected) $\rho$ is shown as well. **Check:** a physical state has $\langle XX\rangle, \langle YY\rangle \le 1$; a
corrected value above 1 means the CM over-corrects (e.g. $X_\pi$ errors in the CM preparation, as in
`0924_3`), and that set's corrected fidelity should not be trusted.

**Simulation with incoherent error.** A Lindblad simulation of the actual sequence — 16 ns, $X_\pi$ on Q1
(100 ns), 26 ns, $\sqrt{i\mathrm{SWAP}}$ as an XY exchange of the snapshot length — with amplitude damping and
pure dephasing on each qubit from the **measured** $T_1$ and $T_2$ echo of the Multi Qubit T1 and T2 run next to
the set (low TLS dips in the repeats removed, see `ba.tls_dropped`), and no gate errors. Two versions are compared with the data:

* **$T_1$, $T_2$ only** – the state right after the $\sqrt{i\mathrm{SWAP}}$, compared with the *corrected* $\rho$;
* **$T_1$, $T_2$ + readout** – the same state taken through the tomography pulses (with decay) and a readout given
  by the set's **measured** mean confusion matrix, compared with the *raw* $\rho$. The CM has all four prepared
  basis states, so it includes the readout crosstalk between the qubits: each qubit's $F_e$ is lower when the
  other qubit is in $|g\rangle$ (for `0924_1`: Q1 0.867 vs 0.883, Q2 0.890 vs 0.903). A single-qubit calibration
  with both qubits excited misses this and underestimates the raw single-qubit Pauli terms, which for a Bell
  state are almost pure readout asymmetry ($\approx F_g - F_e$). The CM also contains the decay during its own
  $X_\pi$ preparation, which this simulation then counts a second time (a small effect, ≲ 1 % on the raw
  correlators).

In [ ]:
import sys
sys.path.insert(0, 'bell_state')
import bell_analysis as ba

BELL_SET = 'Bell_state_0924_1'              # bell_state/<BELL_SET>.csv and <BELL_SET>_confusion.csv
T1T2_TUID = '20260924-174141-671-4869f0'   # Multi Qubit T1 and T2 measured right before the set

bell = ba.BellSet(BELL_SET)

# measured, pooled: per-run CM correction -> average rho_lin -> one MLE; bootstrap over runs
rho_meas = bell.pooled_rho()
F_meas, F_meas_err = bell.pooled_F()
# raw: no readout correction (linear inversion of the pooled counts)
rho_raw = ba.lin_rho(bell.P.mean(0), np.eye(4))
F_raw = ba.fidelity(rho_raw)
# previous estimator, for reference only (biased low)
F_old = bell.per_run_F().mean()
# over-correction check on the pooled, corrected rho before the MLE
s_lin = compute_stokes(bell.lin.mean(0))

print(f'{BELL_SET}: {len(bell)} runs, {bell.times[0]:%H:%M}-{bell.times[-1]:%H:%M}, '
      f'{bell.n_tomo} shots/setting, CM {bell.cm_source}')
print(f'corrected <XX> = {s_lin[4]:.4f}, <YY> = {s_lin[9]:.4f}, <ZZ> = {s_lin[14]:.4f}'
      + ('   ** > 1: CM over-corrects, do not trust F **' if max(s_lin[4], s_lin[9]) > 1 else '   (physical)'))

In [ ]:
# simulated state with incoherent error only: measured T1 and T2 echo of each qubit (TLS dips removed)
coh = ba.coherence_times(T1T2_TUID)
T1 = [coh[q]['T1'][0] * 1e-6 for q in ('Q1', 'Q2')]
T2 = [coh[q]['T2echo'][0] * 1e-6 for q in ('Q1', 'Q2')]
t_1q, t_2q = ba.snapshot_durations(bell.tuids[0])
# measured readout: the set's mean confusion matrix CM[measured, prepared] (all four prepared states,
# so it includes the readout crosstalk between Q1 and Q2)
CM_ro = bell.CM.mean(0)

model = ba.ErrorModel(T1, T2, t_1q, t_2q)
rho_sim = model.state(ba.IDEAL_PARAMS)                          # T1, T2echo only
p_true = model.probs(ba.IDEAL_PARAMS)                           # after tomography pulses + decay, ideal readout
rho_sim_raw = ba.lin_rho(p_true @ CM_ro.T, np.eye(4))          # + measured readout (CM_ro @ p per setting)
F_sim, F_sim_raw = ba.fidelity(rho_sim), ba.fidelity(rho_sim_raw)

tls = ba.tls_dropped(T1T2_TUID)
for q in ('Q1', 'Q2'):
    removed = '; '.join(f'{k} {np.round(v, 1).tolist()}' for k, v in tls[q].items() if len(v))
    print(q, ', '.join(f"{k} = {coh[q][k][0]:.1f} ± {coh[q][k][1]:.1f} us" for k in ('T1', 'T2echo'))
          + (f'   (TLS dips removed: {removed})' if removed else ''))
c = CM_ro
print(f'readout: mean CM of the {len(bell)} runs;  F_e(Q1) = {c[2,2]+c[3,2]:.4f} with Q2 in g, {c[2,3]+c[3,3]:.4f} with Q2 in e;'
      f'  F_e(Q2) = {c[1,1]+c[3,1]:.4f} with Q1 in g, {c[1,3]+c[3,3]:.4f} with Q1 in e')
print(f'pulses: 1Q {t_1q*1e9:.0f} ns, sqrt_iSWAP {t_2q*1e9:.0f} ns (+ {ba.T_BUF_2Q*1e9:.0f} ns)\n')
print(f"{'simulated, T1 + T2echo':44s}{F_sim:>9.4f}")
print(f"{'measured, corrected (pooled)':44s}{F_meas:>9.4f} ± {F_meas_err:.4f}")
print(f"{'simulated, T1 + T2echo + readout':44s}{F_sim_raw:>9.4f}")
print(f"{'measured, raw (no readout correction)':44s}{F_raw:>9.4f}")
print(f"{'per-run MLE average (biased, reference)':44s}{F_old:>9.4f}")

In the Pauli plot the simulation ($T_1$, $T_2$ echo) is drawn as a vertical line down the centre of each bar: $T_1$, $T_2$ only on the
corrected bar, $T_1$, $T_2$ + readout on the raw bar. The density-matrix plot overlays the raw (solid) and
corrected (transparent) measurement.
For `0924_1` the corrected, pooled fidelity agrees with the $T_1$/$T_2$ simulation: the remaining infidelity is
incoherent, and gate-calibration errors are below the resolution. The raw bars are smaller by the readout
contrast. With the measured CM as the readout, the $T_1$, $T_2$ + readout simulation reproduces the Z-basis terms
(the single-qubit $Z$ terms and $\langle ZZ\rangle$) and most of the positive single-qubit terms that asymmetric,
crosstalking readout produces; it stays ≈ 0.01 below the data in $\langle XX\rangle$, $\langle YY\rangle$ and
Q1's $X$/$Y$ terms, partly because the CM's own preparation decay is counted twice.

In [ ]:
labels = ['Z\nI', 'X\nI', 'Y\nI', 'I\nZ', 'I\nX', 'I\nY',
          'Z\nZ', 'Z\nX', 'Z\nY', 'X\nZ', 'X\nX', 'X\nY', 'Y\nZ', 'Y\nX', 'Y\nY']   # top: Q2, bottom: Q1
pauli = lambda rho: rearrange_stokes(compute_stokes(rho))
x = np.arange(15)
w = 0.4
LIGHT_BLUE, DARK_BLUE = '#8fc3e8', C['Q1']
LIGHT_ORANGE, DARK_ORANGE = '#f6a96b', C['Q2']
# (offset, measured label, measured values, bar colour, simulated label, simulated values, line colour)
series = [(-w / 2, 'Exp. raw', pauli(rho_raw), LIGHT_BLUE,
           'Sim. incoherent+RO', pauli(rho_sim_raw), LIGHT_ORANGE),
          (+w / 2, 'Exp. corrected', pauli(rho_meas), DARK_BLUE,
           'Sim. incoherent', pauli(rho_sim), DARK_ORANGE)]

with plt.rc_context(PRA):
    fig, ax = plt.subplots(figsize=(3.375, 2.1))          # PRA single column
    for sep in (2.5, 5.5):
        ax.axvline(sep, color='0.8', lw=0.6)
    ax.bar(x, pauli(ba.IDEAL), 2 * w, facecolor='none', edgecolor='black', lw=0.6, label='Ideal', zorder=3)
    for dx, mlab, meas, bcol, slab, sim, lcol in series:
        ax.bar(x + dx, meas, w, color=bcol, label=mlab, zorder=2)
    for dx, mlab, meas, bcol, slab, sim, lcol in series:
        # simulation as a vertical line from 0 to the simulated value, down the centre of its bar
        ax.vlines(x + dx, 0, sim, color=lcol, lw=1.0, zorder=3, label=slab)
    ax.axhline(0, color='black', lw=0.5, zorder=4)
    ax.set_xticks(x, labels, fontsize=6)
    ax.tick_params(axis='x', length=2, pad=1.5)
    ax.set_xlim(-0.6, 14.6)
    ax.set_ylim(-1.1, 1.1)
    ax.set_yticks([-1, 0, 1])
    ax.set_ylabel(r'$\langle P \rangle$')
    for xc, t in ((1, 'Q2'), (4, 'Q1'), (10, 'Q1Q2')):
        ax.text(xc, -0.93, t, ha='center', fontsize=7)
    h, l = ax.get_legend_handles_labels()
    order = [l.index(k) for k in ('Ideal', 'Exp. raw', 'Exp. corrected', 'Sim. incoherent+RO', 'Sim. incoherent')]
    ax.legend([h[i] for i in order], [l[i] for i in order], loc='lower left', bbox_to_anchor=(-0.02, 1.0),
              ncol=2, frameon=False, fontsize=6, handlelength=1.2, handletextpad=0.4, columnspacing=1.5,
              borderaxespad=0.2)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, f'bell_state_pauli.{ext}'), bbox_inches='tight', pad_inches=0.02)
    plt.show()

In [ ]:
import qutip as qt

basis = [r'$|00\rangle$', r'$|01\rangle$', r'$|10\rangle$', r'$|11\rangle$']

with plt.rc_context(PRA):
    fig = plt.figure(figsize=(3.375, 2.6))                   # fits a PRA single column (3.375 in)
    ax = fig.add_axes([-0.04, 0.0, 0.86, 1.0], projection='3d')
    # corrected: transparent bars (with the phase colorbar); raw: solid bars drawn inside them
    qt.visualization.matrix_histogram(
        qt.Qobj(rho_meas, dims=[[2, 2], [2, 2]]), basis, basis, bar_style='abs', color_style='phase',
        ax=ax, fig=fig, cmap=plt.cm.bwr, options={'bars_alpha': 0.1, 'bars_spacing': 0.3, 'bars_lw': 0.2})
    qt.visualization.matrix_histogram(
        qt.Qobj(rho_raw, dims=[[2, 2], [2, 2]]), basis, basis, bar_style='abs', color_style='phase',
        ax=ax, fig=fig, cmap=plt.cm.bwr, colorbar=False,
        options={'bars_alpha': 0.75, 'bars_spacing': 0.3, 'bars_lw': 0})
    ax.set_zticks([0, 0.25, 0.5])
    ax.set_xticklabels(basis, fontsize=7)
    ax.set_yticklabels(basis, fontsize=7)
    ax.tick_params(axis='z', labelsize=7)

    cbar_ax = fig.axes[-1]                                   # the colorbar made by the first call
    cbar_ax.set_position([0.76, 0.25, 0.022, 0.5])
    cbar_ax.set_ylabel('Phase')
    cbar_ax.tick_params(labelsize=7)
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, f'bell_state_rho.{ext}'), bbox_inches='tight', pad_inches=0.02)
    plt.show()

### 5.2 Process Tomography

Repeated process tomography of the iSWAP (Clifford 5760): every run measures all 36 × 36 (prepare,
measure) settings with 400 shots, each setting repeated without the gate (identity reference). The
analysis is the one of `process_tomography/` (`qpt_analysis.py`; details in `qpt_details.ipynb` §7 and
`qpt_trials.ipynb`):

1. **Pool the runs.** $P_{00}$ of every setting, gate and reference, averaged over the runs, then one fit.
   Everything before the fit is linear, so this is exact; one fit per run would be biased low by the
   few shots (400) per setting.
2. **Readout in the model.** The "00" effect of the forward model is $F^\dagger\,\mathrm{diag}(M_{00})\,F$, with
   $M_{00}$ from the measured $F_g$, $F_e$ of each run (IQ shots of the readout calibration in force for it).
3. **SPAM correction.** Maximum-likelihood fit of the reference $E$, then of the gate $G$ with
   $E^{1/2} G E^{1/2}$ in the forward model — CPTP by construction.
4. **Error bars:** bootstrap over runs (runs resampled with replacement, pooled and refitted).

Angles are those of the closest unitary $U_{\rm fit}$ in the iSWAP_RPE parametrization with $\varphi_p = 0$
($\varphi_p$ is not observable at a full iSWAP: $\theta_1$, $\theta_2$ depend on that convention, $\theta_{\rm sum}$ and
$\theta_d$ do not). Without the readout in the model (step 2) the SPAM correction assigns about 2 % of
readout error to the gate and shifts $\varphi_{zz}$ by about −3° (5.2.1).

**Another data set:** edit the settings cell below — the tuid list, the T1/T2 run for 5.2.1 (or `None` to
skip the simulation) and `N_BOOT` (0 = no error bars, for a quick look) — and run 5.2. Fits are cached per
tuid list in `process_tomography/cache/`: the three trials of `qpt_trials.ipynb` reuse their caches, any
other list is fitted once (a few minutes) under a name built from its first and last tuid.

In [ ]:
# ---- settings of 5.2 ----
# trial 1 (2026-09-24, 20 runs; T1/T2 run '20260924-223804-659-6021ad', N_BOOT = 24):
# QPT_tuid_list = ['20260924-2103', '20260924-2108', '20260924-2113', '20260924-2118', '20260924-2123', '20260924-2128', '20260924-2133', '20260924-2138', '20260924-2143', '20260924-2148', '20260924-2153', '20260924-2158', '20260924-2203', '20260924-2208', '20260924-2213', '20260924-2218', '20260924-2223', '20260924-2228', '20260924-2233', '20260924-2238']
# QPT_tuid_list = ['20260930-1740', '20260930-1745', '20260930-1750', '20260930-1755', '20260930-1759', ]
QPT_tuid_list = ['20260930-185954', '20260930-190445', '20260930-191001', '20260930-191452']
QPT_T1T2_TUID = None   # Multi Qubit T1 and T2 run(s) next to the set, for 5.2.1; None = skip
N_BOOT = 0             # bootstrap resamples over runs for the error bars; 0 = no error bars (quick look)

In [ ]:
# Repeated QPT set -> pooled P00 -> readout-aware MLE (process_tomography/qpt_analysis.py).
# Fits are cached in process_tomography/cache/<QPT_NAME>_*.npz, one name per tuid list.
sys.path.insert(0, 'process_tomography')
import qpt_analysis as qa
import run_qpt_trials as rt

# a trial of process_tomography/qpt_trials.ipynb keeps its name (and cache); any other list gets its own
QPT_NAME = next((name for name, (t, _) in rt.TRIALS.items() if list(t) == list(QPT_tuid_list)),
                f'QPT_{QPT_tuid_list[0]}_{QPT_tuid_list[-1]}_n{len(QPT_tuid_list)}')
print('cache name:', QPT_NAME)

qs = qa.QPTSet(QPT_tuid_list)
p00, ref_p00 = qs.pooled()          # mean over runs: gate, identity reference
n_shots = qs.n_shots()
readout = qs.readout()              # (run, qubit, [F_g, F_e])
ro = readout.mean(0)
print(f'{len(qs)} runs x {qs.n_rep} shots = {n_shots} shots per setting, '
      f'{qs.times[0]:%Y-%m-%d %H:%M}-{qs.times[-1]:%H:%M}')
print(f'readout (mean over runs): Q1 F_g = {ro[0, 0]:.4f}, F_e = {ro[0, 1]:.4f};  '
      f'Q2 F_g = {ro[1, 0]:.4f}, F_e = {ro[1, 1]:.4f}')

cache = lambda label: os.path.join(qa.CACHE, f'{QPT_NAME}_{label}.npz')
rec = qa.cached_reconstruct(cache('meas_ra'), p00, ref_p00, n_shots, forward=qa.readout_forward(readout))
rec_ideal_ro = qa.cached_reconstruct(cache('meas'), p00, ref_p00, n_shots)   # ideal "00" effect, for comparison
# bootstrap over runs (parallel fits, a few minutes when not cached)
if N_BOOT:
    qa.run_jobs(qa.bootstrap_jobs(QPT_NAME, qs, readout, N_BOOT), workers=20)
boot = [qa.load_recon(p) for p in qa.bootstrap_paths(QPT_NAME, N_BOOT)]

corrected_chi_Qobj = rec['corr']    # SPAM-corrected gate
measured_chi_Qobj = rec['raw']      # uncorrected gate
reference_chi_Qobj = rec['ref']     # identity reference = SPAM map E

In [ ]:
# Physicality report: the fit is CPTP by construction, so both numbers should come out clean.
J = qt.to_choi(corrected_chi_Qobj).full()
w_choi = np.linalg.eigvalsh((J + J.conj().T) / 2)
tp_violation = np.linalg.norm(np.einsum('ikjk->ij', J.reshape(4, 4, 4, 4)) - np.eye(4))
print(f"SPAM-corrected Choi eigs:  min {w_choi.min():+.4f}  max {w_choi.max():.4f}")
print(f"Trace-preservation check:  ||Tr_out(J) - I|| = {tp_violation:.2e}")

In [ ]:
fig, ax = qpt_plot_cmap(corrected_chi_Qobj / 16, op_label, 'iSWAP, SPAM-corrected (readout-aware)',
                       cmap=plt.cm.bwr, figsize=(10, 8), threshold=0.00)
ax.set_zticks([0, 0.125, 0.25])
ax.view_init(elev=30, azim=-35)
ax.tick_params(axis='x', labelsize=10)   # the II, IX, IY... labels
ax.tick_params(axis='y', labelsize=10)
ax.tick_params(axis='z', labelsize=12)   # the height (0.0, 0.2...) numbers
fig.savefig(os.path.join(FIGDIR, 'qpt_chi.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Fitted angles: closest unitary U_fit of the SPAM-corrected gate as iSWAP_RPE, phi_p = 0.
# Errors: standard deviation over the bootstrap resamples (none with N_BOOT = 0).
boot_sd = lambda xs: np.std(xs, ddof=1) if len(xs) > 1 else np.nan
pm = lambda e, f: '' if np.isnan(e) else f' ± {e:{f}}'
U_fit = rec['U_fit']
ang = qa.summarize(rec)['angles_deg']
ang_boot = [qa.summarize(b)['angles_deg'] for b in boot]
ang_fit, _, _ = fit_iswap_rpe_to_chi(corrected_chi_Qobj)    # direct least-squares fit of iSWAP_RPE to chi
ang_fit_d = deg(ang_fit)

print(f"{'iSWAP_RPE angles (deg), phi_p = 0':34s}{'U_fit':>16s}{'chi fit':>10s}")
for k in ['theta_p', 'phi_zz', 'theta_sum', 'theta_d', 'theta_s', 'theta_1', 'theta_2']:
    sd = boot_sd([a[k] for a in ang_boot])
    print(f'   {k:31s}{ang[k]:9.2f}{pm(sd, "4.2f"):>7s}{ang_fit_d[k]:10.2f}')


def U_zz(U):
    # perfect iSWAP whose only coherent error is the ZZ phase of U: iSWAP_RPE(90 deg, 0, 0, 0, phi_zz)
    return iSWAP_RPE([np.pi / 2, 0, 0, 0, extract_angles(qa.as_unitary(U))['phi_zz']])


U_iSWAP = qt.Qobj(qa.U_ISWAP, dims=[[2, 2], [2, 2]])
targets = [   # each bootstrap resample is compared with its own fitted target
    ('closest unitary U_fit (incoherent error)', lambda r: r['U_fit']),
    ('iSWAP with the fitted phi_zz', lambda r: U_zz(r['U_fit'])),
    ('ideal iSWAP', lambda r: U_iSWAP),
]
print(f"\n{'process fidelity of the SPAM-corrected gate to':46s}{'F_pro':>17s}{'F_avg':>17s}")
qpt_F = {}
for lab, target in targets:
    F = proc_fid_to_unitary(corrected_chi_Qobj, target(rec))
    e = boot_sd([proc_fid_to_unitary(b['corr'], target(b)) for b in boot])
    qpt_F[lab] = (F, e)
    print(f'{lab:46s}{F:10.4f}{pm(e, ".4f"):>9s}{qa.avg_gate_fidelity(F):10.4f}{pm(0.8 * e, ".4f"):>9s}')

a0 = qa.summarize(rec_ideal_ro)['angles_deg']
print(f"\nwithout the readout in the model (ideal '00' effect): F_pro = "
      f"{proc_fid_to_unitary(rec_ideal_ro['corr'], rec_ideal_ro['U_fit']):.4f},  phi_zz = {a0['phi_zz']:.2f} deg")

In [ ]:
# Pauli transfer matrix of the SPAM-corrected gate vs its closest unitary; the difference is the incoherent error
R_exp, R_fit = qa.ptm(corrected_chi_Qobj), qa.ptm(qt.to_chi(U_fit))
panels = [('Experiment', R_exp, 1), (r'Closest unitary $U_{\rm fit}$', R_fit, 1), (r'Exp. $-$ $U_{\rm fit}$', R_exp - R_fit, 0.1)]
with plt.rc_context(PRA):
    fig, axs = plt.subplots(1, 3, figsize=(7.0, 2.55), constrained_layout=True)   # PRA double column
    for i, (ax, (title, R, lim)) in enumerate(zip(axs, panels)):
        im = ax.imshow(R, cmap='RdBu_r', vmin=-lim, vmax=lim)
        ax.set_title(title)
        ax.set_xticks(range(16), qa.PAULI_LABELS, rotation=90, fontsize=5)
        ax.set_yticks(range(16), qa.PAULI_LABELS if i == 0 else [], fontsize=5)
        ax.tick_params(length=0, top=False, right=False)
        if i >= 1:
            fig.colorbar(im, ax=ax, shrink=0.8, ticks=[-lim, 0, lim])
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, f'qpt_ptm_fit.{ext}'), bbox_inches='tight', pad_inches=0.02)
    plt.show()

#### 5.2.1 Incoherent error: simulation

A Lindblad simulation of the QPT sequence — reset | 16 ns | prep rotation (100 ns) | 26 ns | iSWAP (494 ns) |
16 ns | meas rotation (100 ns) | readout, identity rotations padded to the same length — with amplitude damping
and pure dephasing on each qubit from the **measured** $T_1$ and $T_2$ echo of the Multi Qubit T1 and T2 run right
after the set, and no gate errors other than the ZZ phase: the simulated gate is iSWAP_RPE with
$\theta_p = 90°$, $\theta_1 = \theta_2 = \varphi_p = 0$ and the $\varphi_{zz}$ fitted above. Every fidelity is to the closest
unitary of its own reconstruction, so only incoherent error enters.

The simulated $P_{00}$ of all 1296 settings, gate and identity reference, with the **measured** $F_g$, $F_e$ of each
run and 400 shots per run, go through both reconstructions of the data: readout-aware (as above) and with the
ideal "00" effect.

* **$T_1$, $T_2$ echo only** limit the gate itself to $F_{\rm pro} \approx 0.985$ ($F_{\rm avg}$ 0.988). Through tomography
  and the reconstruction with perfect readout the same value comes back: decay during the tomography
  rotations is removed correctly by the reference.
* **With the readout in the model** the simulation returns that limit, and the experiment is about 0.9 %
  (process infidelity) below it: that part of the error is beyond the measured coherence.
* **With the ideal "00" effect** the simulation reproduces the measured 0.963: readout error acts after the
  measurement rotation, so no process in the gate's position describes it, and the $E^{1/2} G E^{1/2}$ correction
  assigns about 2 % of it to the gate.

In [ ]:
# measured and simulated reconstructions (cached in process_tomography/cache/; redo=True refits, ~4 min)
# QPT_T1T2_TUID is set in the settings cell of 5.2; None skips 5.2.1
if QPT_T1T2_TUID is None:
    qpt = None
    print('5.2.1 skipped: QPT_T1T2_TUID = None')
else:
    qpt = qa.analyze(QPT_NAME, QPT_tuid_list, QPT_T1T2_TUID, verbose=False)
    qa.report(qpt)

In [ ]:
# process infidelity at each stage of the simulation vs the measurement, for both reconstructions
if qpt is not None:
    Fg = qpt['F_groups']
    F_err = Fg.std(ddof=1) / np.sqrt(len(Fg)) if len(Fg) > 1 else None               # ideal "00" effect: sem of the readout-calibration groups
    F_err_ra = qpt_F['closest unitary U_fit (incoherent error)'][1]
    F_err_ra = None if np.isnan(F_err_ra) else F_err_ra   # readout-aware: bootstrap over runs
    s = qpt['sim_summary']
    rows = [  # (label, process infidelity, error, kind)
        ('Sim.: $T_1$, $T_2$ echo (gate)',       1 - qpt['F_model'],                       None,     'sim'),
        ('Sim.: + tomography',                   1 - s['sim_decay']['F_corr'],              None,     'sim'),
        ('Sim.: + readout, readout-aware',       1 - s['sim_decay_ro_shots_ra']['F_corr'],  None,     'sim'),
        ('Experiment, readout-aware',            1 - qpt['meas_ra_summary']['F_corr'],      F_err_ra, 'exp'),
        ('Sim.: + readout, ideal "00" effect',   1 - s['sim_decay_ro_shots']['F_corr'],     None,     'sim'),
        ('Experiment, ideal "00" effect',        1 - qpt['meas_summary']['F_corr'],         F_err,    'exp'),
    ]
    with plt.rc_context(PRA):
        fig, ax = plt.subplots(figsize=(3.375, 1.8))          # PRA single column
        y = np.arange(len(rows))[::-1]
        for yi, (lab, r, e, kind) in zip(y, rows):
            col = C['Q2'] if kind == 'sim' else C['Q1']
            ax.hlines(yi, 0, 100 * r, color=col, lw=1.0)
            ax.errorbar(100 * r, yi, xerr=None if e is None else 100 * e, fmt='o', ms=4, color=col,
                        mec='white', mew=0.6, capsize=2, elinewidth=0.8, zorder=3)
            ax.text(100 * r + 0.12, yi + 0.18, f'{100 * r:.2f}%', fontsize=6, va='bottom', color='0.25')
        ax.set_yticks(y, [r[0] for r in rows])
        ax.set_xlim(0, 4.6)
        ax.set_ylim(-0.6, len(rows) - 0.4)
        ax.set_xlabel(r'Process infidelity $1 - F_{\rm pro}$ (%)')
        for ext in ('pdf', 'png'):
            fig.savefig(os.path.join(FIGDIR, f'qpt_infidelity.{ext}'), bbox_inches='tight', pad_inches=0.02)
        plt.show()

In [ ]:
# Pauli transfer matrix: measured vs simulated (T1, T2 echo + readout), both readout-aware
if qpt is not None:
    R_meas = qa.ptm(qpt['meas_ra']['corr'])
    R_sim = qa.ptm(qpt['sims']['sim_decay_ro_ra']['corr'])
    panels = [('Experiment', R_meas), ('Simulation', R_sim), ('Exp. $-$ sim.', R_meas - R_sim)]
    with plt.rc_context(PRA):
        fig, axs = plt.subplots(1, 3, figsize=(7.0, 2.55), constrained_layout=True)   # PRA double column
        for i, (ax, (title, R)) in enumerate(zip(axs, panels)):
            lim = 1 if i < 2 else 0.1
            im = ax.imshow(R, cmap='RdBu_r', vmin=-lim, vmax=lim)
            ax.set_title(title)
            ax.set_xticks(range(16), qa.PAULI_LABELS, rotation=90, fontsize=5)
            ax.set_yticks(range(16), qa.PAULI_LABELS if i == 0 else [], fontsize=5)
            ax.tick_params(length=0, top=False, right=False)
            if i >= 1:
                fig.colorbar(im, ax=ax, shrink=0.8, ticks=[-lim, 0, lim])
        for ext in ('pdf', 'png'):
            fig.savefig(os.path.join(FIGDIR, f'qpt_ptm.{ext}'), bbox_inches='tight', pad_inches=0.02)
        plt.show()
    print(f'max |R_meas - R_sim| = {np.abs(R_meas - R_sim).max():.4f}')

## 6. Theory figures (from the sim notebook on the other machine)

The circuit-model figures (mode frequencies + $\zeta_{ZZ}$, the $\zeta_{ZZ}$ landscape,
$g_{\rm eff}$/Stark-shifted pump frequency, and iSWAP gate time vs flux) are generated by
`SNAIL_315_plots.ipynb`, which lives on the other computer and is only reachable here
through the synced Drive path:

```
C:\Users\1238Tech\Google Drive Streaming\Other computers\My MacBook Pro\Research\Palmer Lab\Python Codes\SNAIL Two Qubit Gate\Zhuo's sim\SNL315\SNAIL_315_plots.ipynb
```

That notebook's own fit and Hamiltonian model (hundreds of lines, plus `SNL315_fit_075.json`
and the `results/`, `gate_715/`, `bias_sweep/` data it depends on) are not duplicated here --
only its finished output PDFs/PNGs, from its `paper_figs/` folder, are copied into this
notebook's `paper_plots/theory/`. Re-run `SNAIL_315_plots.ipynb` on the other machine first
if the theory figures need updating, then re-run the cell below to pull the new copies in.

In [ ]:
import shutil

# REMOTE_SIM_DIR is defined once in the setup cell above.
REMOTE_PAPER_FIGS = os.path.join(REMOTE_SIM_DIR, 'paper_figs')

THEORY_FIGDIR = os.path.join(FIGDIR, 'theory')
os.makedirs(THEORY_FIGDIR, exist_ok=True)

copied = []
for fname in sorted(os.listdir(REMOTE_PAPER_FIGS)):
    if fname.endswith(('.pdf', '.png')):
        shutil.copy2(os.path.join(REMOTE_PAPER_FIGS, fname), os.path.join(THEORY_FIGDIR, fname))
        copied.append(fname)

print('copied %d files from %s\n  -> %s' % (len(copied), REMOTE_PAPER_FIGS, THEORY_FIGDIR))
for fname in copied:
    print(' ', fname)

In [ ]:
# Combined figure: iSWAP population chevrons (top, from this notebook's dataset --
# reuses x0_data/x1_data/y0_data/y1_data loaded in section 1) over g_eff and the
# Stark-shifted pump frequency vs pump power (bottom, plotting code ported from
# SNAIL_315_plots.ipynb Figure 3, data read from its results/ folder via REMOTE_SIM_DIR).
# Both rows use the one shared PRA style and colour palette (C, GRID). The figure is split
# into two subfigures (fig.subfigures) rather than one shared set of manual rectangles --
# giving the top subfigure exactly section 1's own figsize (3.375, 1.7) and exactly its
# LEFT/RIGHT/BOTTOM/TOP fractions reproduces its panels at the identical size in inches,
# which plain fraction-of-one-figure math could only ever approximate.
from matplotlib.ticker import MultipleLocator

with open(os.path.join(REMOTE_SIM_DIR, 'results', 'paper_fig_geff.json')) as f:
    D = json.load(f)

P     = np.array(D['P_mW'])
g_exp = 1e3 * np.array(D['g_meas_GHz'])          # MHz
g_sim = 1e3 * np.array(D['g_sim_GHz'])
g_eq  = 1e3 * np.array(D['g_eq_fit_GHz'])        # attenuation-fitted, not the raw Eq. 55
f_exp = 1e3 * np.array(D['f_meas_GHz'])
f_sim = 1e3 * np.array(D['f_sim_GHz'])
EQ_LABEL = 'Eq.'          # <-- set the equation number here once it is assigned

FIG_W = 3.375
TOP_H = 1.7     # identical to section 1's own figsize height
BOT_H = 1.7

with plt.rc_context(PRA):
    fig = plt.figure(figsize=(FIG_W, TOP_H + BOT_H))
    subfig_top, subfig_bot = fig.subfigures(2, 1, height_ratios=[TOP_H, BOT_H])

    # ---- top: section 1's exact layout, reproduced at its exact size ------------
    LEFT, RIGHT, BOTTOM, TOP = 0.135, 0.80, 0.24, 0.96      # identical to cell 2
    axes = subfig_top.subplots(1, 2, sharey=True, gridspec_kw={'wspace': 0.07})
    subfig_top.subplots_adjust(left=LEFT, right=RIGHT, bottom=BOTTOM, top=TOP)

    pcm = draw_iswap_chevron(axes[0], axes[1], x0_data, x1_data, y0_data, y1_data)
    subfig_top.supxlabel('Pump duration (\u03bcs)', x=0.5 * (LEFT + RIGHT), y=0.06)

    cax = subfig_top.add_axes([RIGHT + 0.025, BOTTOM, 0.025, TOP - BOTTOM])
    cb = subfig_top.colorbar(pcm, cax=cax, ticks=[0, 0.5, 1])
    cb.ax.set_yticklabels(['0', '0.5', '1'])
    cb.set_label('Population')
    cb.outline.set_linewidth(0.7)
    cb.ax.tick_params(which='both', direction='out', color='black', left=False, right=True)

    # ---- bottom: g_eff and Stark-shifted pump frequency vs power ----------------
    ax_g, ax_f = subfig_bot.subplots(1, 2)
    subfig_bot.subplots_adjust(left=0.12, right=0.935, bottom=0.30, top=1.0, wspace=0.45)

    ax_g.plot(P, g_exp, 'o', ms=3.0, color=C['Q1'], mec='none', label='Exp.', zorder=3)
    ax_g.plot(P, g_sim, '-', color='black', lw=0.9, label='Sim.', zorder=4)
    ax_g.plot(P, g_eq, '--', color=C['Q2'], lw=0.9, label=EQ_LABEL, zorder=2)
    ax_g.set_ylabel(r'$g_{\mathrm{eff}}/2\pi$ (MHz)', labelpad=1)
    ax_g.set_ylim(0, 1.3)
    ax_g.yaxis.set_major_locator(MultipleLocator(0.5))

    ax_f.plot(P, f_exp, 'o', ms=3.0, color=C['Q1'], mec='none', label='Exp.', zorder=3)
    ax_f.plot(P, f_sim, '-', color='black', lw=0.9, label='Sim.', zorder=4)
    ax_f.set_ylabel(r'$\omega_{\mathrm{iSWAP}}/2\pi$ (MHz)', labelpad=1)
    ax_f.yaxis.set_major_locator(MultipleLocator(1.0))

    for ax, loc in ((ax_g, 'lower right'), (ax_f, 'upper left')):
        ax.set_xlim(0, P.max() * 1.02)
        ax.xaxis.set_major_locator(MultipleLocator(10))
        leg = ax.legend(loc=loc, fontsize=6.5, handlelength=1.1, handletextpad=0.4,
                        borderpad=0.25, labelspacing=0.18, borderaxespad=0.4,
                        markerscale=0.85, frameon=True, facecolor='white',
                        edgecolor=GRID, framealpha=1.0)
        leg.set_zorder(6)
        leg.get_frame().set_linewidth(0.6)
    subfig_bot.supxlabel('Pump power (mW)', y=0.14)

    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(FIGDIR, 'chevron_and_geff_combined.%s' % ext),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()